In [2]:
# ============================
# Cell 1: Load libraries and preprocessed UAV data
# ============================

# Import NumPy
# Used for working with arrays and loading .npy files
import numpy as np

# Import Pandas
# Used for reading CSV files and handling tables/dataframes
import pandas as pd

# Import TensorFlow
# Main deep learning framework used to build/train neural networks
import tensorflow as tf

# Import Keras from TensorFlow
# Provides simpler functions to build neural network models
from tensorflow import keras


# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,        # measures overall prediction correctness
    precision_score,       # measures how accurate positive predictions are
    recall_score,          # measures how many real positives were found
    f1_score,              # balance between precision and recall
    classification_report  # prints a detailed performance report
)


# Import JSON library
# Used to save or load data in JSON format
import json

# Import time library
# Used to measure how long model training takes
import time

# Import matplotlib
# Used for plotting graphs/charts
import matplotlib


# Use 'Agg' backend
# Useful when saving plots instead of showing them
# Prevents display errors without GUI (especially on Mac)
matplotlib.use('Agg')

# Import pyplot for creating graphs and visualizations
import matplotlib.pyplot as plt


# Print TensorFlow version
# Helps confirm installation and compatibility
print(f"TensorFlow version: {tf.__version__}")


# ============================
# Define file path
# ============================

# Path where preprocessed files are saved
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV-Cyber-Physical/processed/"


# ============================
# Load feature arrays
# ============================

# Load training feature array
# np.load() is faster than reading CSV for large arrays
X_train = np.load(save_path + "X_train.npy")

# Load testing feature array
X_test = np.load(save_path + "X_test.npy")


# ============================
# Load labels
# ============================

# Read training labels from CSV file
# squeeze() converts DataFrame into a 1D Series
#
# Example:
# Before: [[0], [1], [2]]
# After: [0, 1, 2]
y_train = pd.read_csv(save_path + "y_train.csv").squeeze()

# Read testing labels
y_test = pd.read_csv(save_path + "y_test.csv").squeeze()


# ============================
# Load class names
# ============================

# Read label class names from CSV
# squeeze() converts to 1D list
# tolist() converts Pandas object into a normal Python list
#
# Example:
# ['Normal', 'Attack_Type1', 'Attack_Type2']
#
# Used later for classification reports
le_classes = pd.read_csv(
    save_path + "label_classes.csv"
).squeeze().tolist()


# ============================
# Load feature names
# ============================

# Read feature names from CSV file
# Converts them into a Python list
#
# Example:
# ['packet_size', 'signal_strength', 'speed']
#
# Used later for:
# - SHAP
# - Feature Importance
# - Permutation Importance
feature_names = pd.read_csv(
    save_path + "feature_names.csv"
).squeeze().tolist()


# ============================
# Print dataset information
# ============================

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

# Print all class names
print(f"Classes: {le_classes}")

# Print number of features
print(f"Features: {len(feature_names)}")

# Print confirmation message
print("\nData loaded!")

TensorFlow version: 2.16.2
X_train: (23171, 37)
X_test: (9931, 37)
Classes: ['DoS attack', 'Replay', 'benign']
Features: 37

Data loaded!


In [12]:
# Cell 2: filter benign traffic for autoencoder(UNSUPERVISED learn model) training
# RF and CNN models learn from all labled classes(benign + attack traffic)
# autocoder learns only from benign traffic. 
# reason: it learn how to reconstruct normal traffic, after learn it, when it see a abnormal behavior, it can easy to identity.

# filter benign samples: create a true / false mask; 
# i.g.: y_train: ['benign', 'attack', 'benign']; Result:[True, False, True]; true = kee this row; false= remove this row
benign_label = le_classes.index('benign')  # = 2
benign_mask = y_train == benign_label

# Keep only benign rows from training data
X_benign = X_train[benign_mask]

# print dataset statictics
print(f"Total training samples: {len(X_train)}")
print(f"Benign training samples: {len(X_benign)}")
print(f"Percentage benign: {len(X_benign)/len(X_train)*100:.1f}%")
print("\nBenign filter complete! ")

Total training samples: 23171
Benign training samples: 6597
Percentage benign: 28.5%

Benign filter complete! 


In [13]:
# Cell 3: Build Autoencoder architecture

# Autoencoder has two parts:

# ENCODER:
# Compresses input from 37 features → 16 (bottleneck)

# DECODER:
# Reconstructs data back from 16 → 37

# Structure:
# Input(37) → Dense(32) → Dense(16) → Dense(32) → Output(37)
#              ENCODER      BOTTLENECK    DECODER

print("Building Autoencoder...")


# Get number of input features
input_dim = X_train.shape[1]  # 37 for UAV dataset

print(f"Input dimensions: {input_dim}")


# Input layer
# Receives 37 features
input_layer = keras.layers.Input(shape=(input_dim,))


# ============================
# ENCODER: compress data
# ============================

# Dense(32):
# 32 neurons learn compressed representation
encoded = keras.layers.Dense(
    32,
    activation='relu'
)(input_layer)


# BOTTLENECK layer
# Compresses further from 32 → 16
bottleneck = keras.layers.Dense(
    16,
    activation='relu'
)(encoded)


# ============================
# DECODER: reconstruct data
# ============================

# Expand back from 16 → 32
decoded = keras.layers.Dense(
    32,
    activation='relu'
)(bottleneck)


# Output layer
# Reconstruct back to original 37 features
output_layer = keras.layers.Dense(
    input_dim,
    activation='sigmoid'
)(decoded)


# ============================
# Build model
# ============================

ae_model = keras.Model(
    inputs=input_layer,
    outputs=output_layer
)

# ============================
# Compile model
# ============================

# optimizer='adam'
# Automatically adjusts weights efficiently
#
# loss='mse'
# Mean Squared Error
# Measures reconstruction quality
#
# Lower MSE =
# Better reconstruction =
# More normal-looking traffic
ae_model.compile(
    optimizer='adam',
    loss='mse'
)


# Show model structure
ae_model.summary()

print("\nAutoencoder built!")

Building Autoencoder...
Input dimensions: 37


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 37)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 37)             │         1,221 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,509 (13.71 KB)

 Trainable params: 3,509 (13.71 KB)

 Non-trainable params: 0 (0.00 B)


Autoencoder built!


In [14]:
# ============================
# Cell 4: Train Autoencoder on benign traffic ONLY
# ============================

# KEY CONCEPT:
# Autoencoder is self-supervised
#
# Input = Output
#
# The model learns to reconstruct
# benign traffic through the bottleneck layer.
#
# Example:
# ae_model.fit(X_benign, X_benign)
#
# Input and target are the SAME
# because this is a reconstruction task.

print("Training Autoencoder on benign traffic only...")

print(f"Training on {len(X_benign)} benign samples...")


# Record training start time
start_time = time.time()


# Train Autoencoder
history = ae_model.fit(

    # Input = Output
    # Reconstruction learning task
    X_benign,
    X_benign,

    # Number of full passes through dataset
    # Autoencoder often needs more epochs than CNN
    epochs=20,

    # Number of samples processed at once
    # Smaller batch size works better for smaller datasets
    batch_size=256,

    # Use 10% of benign data for validation
    # Helps monitor overfitting
    validation_split=0.1,

    # Show training progress bar
    verbose=1
)


# Record training end time
end_time = time.time()

# Calculate training time
# round(value, 2) = keep 2 decimal places
ae_train_time = round(end_time - start_time, 2)


print("\nTraining complete!")

print(f"Training time: {ae_train_time} seconds")

Training Autoencoder on benign traffic only...
Training on 6597 benign samples...
Epoch 1/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 1.3474 - val_loss: 1.2870
Epoch 2/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1.1942 - val_loss: 1.0595
Epoch 3/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.9781 - val_loss: 0.8772
Epoch 4/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.8202 - val_loss: 0.7795
Epoch 5/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.7526 - val_loss: 0.7449
Epoch 6/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.7131 - val_loss: 0.7356
Epoch 7/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.6994 - val_loss: 0.7295
Epoch 8/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7081 - val_loss: 0.7254
Epoch 9/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7062 - val_loss: 0.7224
Epoch 10/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6878 - val_loss: 0.7206
Epoch 11/20
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.6878 - val_l

In [15]:
# ============================
# Cell 5: Evaluate Autoencoder using reconstruction error threshold
# ============================

print("Evaluating Autoencoder...")


# ============================
# Step 1: Reconstruct test data
# ============================
# The trained Autoencoder takes input data (X_test),
# passes it through:
# Encoder → Bottleneck → Decoder
# and tries to reconstruct the original input.

X_pred = ae_model.predict(X_test, batch_size=256, verbose=0)


# ============================
# Step 2: Compute reconstruction error (MSE)
# ============================
# For each sample:
# - Compute squared difference between original and reconstructed values
# - Then average across all 37 features

mse = np.mean(np.power(X_test - X_pred, 2), axis=1)

print(f"MSE shape: {mse.shape}")
print(f"MSE mean: {mse.mean():.6f}")
print(f"MSE min:  {mse.min():.6f}")
print(f"MSE max:  {mse.max():.6f}")


# ============================
# Step 3: Define anomaly threshold
# ============================
# Use ONLY benign test samples to determine normal error range
# This ensures threshold represents "normal behavior"

benign_test_mask = y_test == benign_label

# Extract reconstruction errors for benign samples only
# MSE measures how badly the Autoencoder reconstructed the data.
# Small MSE = familiar = normal; Large MSE = unfamiliar = suspicious
benign_mse = mse[benign_test_mask.values]

# Set threshold at 95th percentile of benign errors
# Meaning: 95% of normal samples fall below this value
# Anything above is considered abnormal (attack)
threshold = np.percentile(benign_mse, 95) 
# thedshold mean single line in middle, if cross left is normal, and cross right is attack
# Normal score ----|---- Suspicious score
#              threshold

print(f"\nThreshold (95th percentile): {threshold:.6f}")


# ============================
# Step 4: Make predictions using threshold
# ============================
# If reconstruction error is high → anomaly (attack)
# If reconstruction error is low → normal (benign)

y_pred_binary = (mse > threshold).astype(int)


# Convert true labels into binary format
# attack = 1, benign = 0
y_test_binary = (y_test != benign_label).astype(int)


# ============================
# Step 5: Evaluate model performance
# ============================
acc = accuracy_score(y_test_binary, y_pred_binary)

p = precision_score(
    y_test_binary, y_pred_binary,
    average='weighted',
    zero_division=0
)

r = recall_score(
    y_test_binary, y_pred_binary,
    average='weighted',
    zero_division=0
)

f1 = f1_score(
    y_test_binary, y_pred_binary,
    average='weighted',
    zero_division=0
)


# Print final evaluation results
print(f"\n=== Autoencoder Results (UAV Dataset) ===")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recall:    {r:.4f} ({r*100:.2f}%)")
print(f"F1-Score:  {f1:.4f} ({f1*100:.2f}%)")


# Detailed classification report
print("\nDetailed Report:")
print(classification_report(
    y_test_binary,
    y_pred_binary,
    target_names=['benign', 'attack']
))


# ============================
# Step 6: Visualize reconstruction error distribution
# ============================
plt.figure(figsize=(10, 5))

# Plot attack samples reconstruction error distribution
plt.hist(
    mse[~benign_test_mask.values],
    bins=50,
    alpha=0.7,
    label='Attack traffic',
    color='red'
)

# Plot benign samples reconstruction error distribution
plt.hist(
    mse[benign_test_mask.values],
    bins=50,
    alpha=0.7,
    label='Benign traffic',
    color='blue'
)

# Draw decision threshold line
plt.axvline(
    threshold,
    color='black',
    linestyle='--',
    label=f'Threshold={threshold:.4f}'
)

# Labels and formatting
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Count')
plt.title('Autoencoder Reconstruction Error — UAV Dataset')
plt.legend()
plt.tight_layout()

# Save plot to file
plt.savefig(
    save_path + "ae_reconstruction_error.png",
    dpi=150,
    bbox_inches='tight'
)

plt.show()

print("Plot saved!")

Evaluating Autoencoder...
MSE shape: (9931,)
MSE mean: 0.627503
MSE min:  0.248609
MSE max:  34.789917

Threshold (95th percentile): 2.250115

=== Autoencoder Results (UAV Dataset) ===
Accuracy:  0.3062 (30.62%)
Precision: 0.5920 (59.20%)
Recall:    0.3062 (30.62%)
F1-Score:  0.1916 (19.16%)

Detailed Report:
              precision    recall  f1-score   support

      benign       0.28      0.95      0.44      2828
      attack       0.71      0.05      0.09      7103

    accuracy                           0.31      9931
   macro avg       0.50      0.50      0.27      9931
weighted avg       0.59      0.31      0.19      9931

Plot saved!


/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_68715/2628825709.py:153: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Cell 6: Save Autoencoder model and results

# Save model in keras format
ae_model.save(save_path + "ae_model.keras")
print("Autoencoder model saved!")

# Save threshold for SHAP and Permutation later
np.save(save_path + "ae_threshold.npy", np.array([threshold]))
print(f"Threshold saved: {threshold:.6f}")

# Save results as JSON
ae_results = {
    "model": "Autoencoder",
    "dataset": "UAV-Cyber-Physical",
    "accuracy":  round(float(acc), 4),
    "precision": round(float(p), 4),
    "recall":    round(float(r), 4),
    "f1_score":  round(float(f1), 4),
    "threshold": round(float(threshold), 6),
    "training_time": ae_train_time,
    "epochs": 20,
    "batch_size": 256,
    "classes": ["benign", "attack"]
}

with open(save_path + "ae_results.json", "w") as f:
    json.dump(ae_results, f, indent=4)
print("Autoencoder results JSON saved!")

print(f"\nSummary:")
print(f"  Accuracy:   {acc*100:.2f}%")
print(f"  F1-Score:   {f1*100:.2f}%")
print(f"  Threshold:  {threshold:.6f}")
print(f"  Train time: {ae_train_time}s")

Autoencoder model saved!
Threshold saved: 2.250115
Autoencoder results JSON saved!

Summary:
  Accuracy:   30.62%
  F1-Score:   19.16%
  Threshold:  2.250115
  Train time: 9.28s
